In [1]:

!pip install -q sentencepiece sacrebleu rouge-score gradio==4.44.0 huggingface_hub==0.23.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 51.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 89.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.9/130.9 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 4.1.1 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.23.0 which is incompatible.
data

In [2]:

import os
import re
import math
import random
import unicodedata
import glob
from pathlib import Path

import numpy as np
import pandas as pd

# torch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
import sentencepiece as spm

# for checking how good our model is
import sacrebleu
from rouge_score import rouge_scorer


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('gpu')
else:
    print('no gpu')

# make folders for saving stuff later
working_dir = Path.cwd()
save_folder = working_dir / 'my_chatbot_stuff'
tokenizer_folder = save_folder / 'tokenizer_files'

save_folder.mkdir(exist_ok=True)
tokenizer_folder.mkdir(exist_ok=True)

Device: cuda
gpu


## Part 1: Loading Data

We downloaded dataset from Kaggle. It's a TSV file with Urdu sentences.


In [6]:

def find_my_data(folder_to_search):
    """search for tsv/csv/txt files"""
    all_files = []
    
    # look for different file types
    all_files += glob.glob(str(folder_to_search / '**/*.tsv'), recursive=True)
    all_files += glob.glob(str(folder_to_search / '**/*.csv'), recursive=True)
    all_files += glob.glob(str(folder_to_search / '**/*.txt'), recursive=True)
    
    if all_files:
        print(f'Found {len(all_files)} file(s)')
        return all_files[0]
    return None

# Try multiple locations (Kaggle, Colab, local)
possible_locations = [
    Path('/kaggle/input'),  # Kaggle - check this first!
    working_dir / 'data',   # Local
    Path('/content/drive/MyDrive/urdu_data')  # Google Drive if mounted
]

my_data_file = None
for data_folder in possible_locations:
    if data_folder.exists():
        print(f'Checking: {data_folder}')
        my_data_file = find_my_data(data_folder)
        if my_data_file:
            print(f'✓ Found dataset: {my_data_file}')
            break
    else:
        print(f'✗ Not found: {data_folder}')

if not my_data_file:
    print('\n⚠ DATA NOT FOUND in any location!')
    print('Will use dummy data for testing')
    print('\nFor Kaggle: Add dataset as input from:')
    print('  https://www.kaggle.com/datasets/muhammadahmedansari/urdu-dataset-20000')

Checking: /kaggle/input
Found 1 file(s)
✓ Found dataset: /kaggle/input/urdu-dataset-20000/final_main_dataset.tsv


In [7]:
# loading the dataset
def load_urdu_data(filepath):
    
    if not filepath:
    
        dummy_urdu = [
            'السلام علیکم آپ کیسے ہیں',
            'وعلیکم السلام میں ٹھیک ہوں',
            'میرا نام احمد ہے',
            'خوشی ہوئی ملکر',
            'آپ کیا کرتے ہیں',
            'میں طالب علم ہوں'
        ] * 400
        return pd.DataFrame({'text': dummy_urdu})
    
    if filepath.endswith('.csv'):
        df = pd.read_csv(filepath, encoding='utf-8')
    elif filepath.endswith('.tsv'):
        df = pd.read_csv(filepath, sep='\t', encoding='utf-8')
    else:
        with open(filepath, encoding='utf-8', errors='ignore') as f:
            lines = [line.strip() for line in f if line.strip()]
        df = pd.DataFrame({'text': lines})
    
    urdu_col = None
    for col in df.columns:
        sample = str(df[col].iloc[0]) if len(df) > 0 else ''
        # check for Urdu characters
        if any('\u0600' <= c <= '\u06FF' for c in sample):
            urdu_col = col
            break
    
    if urdu_col:
        df = pd.DataFrame({'text': df[urdu_col].astype(str)})
    else:
        df = pd.DataFrame({'text': df[df.columns[0]].astype(str)})
    
    # filter out hashes and short text
    df = df[df['text'].str.len() > 5]
    df = df[~df['text'].str.match(r'^[0-9a-f]{30,}$', case=False)]
    
    return df.reset_index(drop=True)

raw_data = load_urdu_data(my_data_file)
print(f'Loaded {len(raw_data)} rows')
raw_data.head(3)

Loaded 19857 rows


,text
0,کبھی کبھار ہی خیالی پلاو بناتا ہوں
1,اور پھر ممکن ہے کہ پاکستان بھی ہو
2,یہ فیصلہ بھی گزشتہ دو سال میں


## Part 2: Text Cleaning

Urdu text has lots of extra marks (diacritics) that we don't need.


In [9]:

DIACRITICS_PATTERN = re.compile(r'[\u064B-\u0652\u0670\u0653-\u065F\u06D6-\u06ED]')
ZERO_WIDTH = re.compile(r'[\u200c\u200d]')  # invisible chars that mess things up
TATWEEL = re.compile(r'\u0640')  # that line thing in Arabic

LETTER_FIXES = {
 
    'أ': 'ا', 'إ': 'ا', 'آ': 'ا', 'ٱ': 'ا',
    # yeh variants
    'ي': 'ی', 'ى': 'ی', 'ئ': 'ی',
    # heh/teh variants
    'ة': 'ہ', 'ۃ': 'ہ', 'ۀ': 'ہ',
    # kafs
    'ك': 'ک',
    # waw
    'ؤ': 'و'
}


PUNCT_FIXES = {
    '،': ',',
    '؛': ';',
    '۔': '.',  
    '؟': '?',  
}


DIGIT_MAP = str.maketrans('۰۱۲۳۴۵۶۷۸۹', '0123456789')

def clean_urdu_text(text):
    """clean and normalize urdu text"""
    if not isinstance(text, str):
        return ''
    
    
    text = unicodedata.normalize('NFKC', text)

    text = ZERO_WIDTH.sub('', text)
    text = TATWEEL.sub('', text)

    text = DIACRITICS_PATTERN.sub('', text)
    

    for old_letter, new_letter in LETTER_FIXES.items():
        text = text.replace(old_letter, new_letter)
    

    for old_punct, new_punct in PUNCT_FIXES.items():
        text = text.replace(old_punct, new_punct)
    

    text = text.translate(DIGIT_MAP)

    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


print('Cleaning text...')
cleaned_data = pd.DataFrame({
    'text': raw_data['text'].astype(str).apply(clean_urdu_text)
})

cleaned_data = cleaned_data[cleaned_data['text'].str.len() > 0].reset_index(drop=True)

print(f'After cleaning: {len(cleaned_data)} rows')
print('\nBefore vs After:')
for i in range(min(3, len(raw_data))):
    print(f'\nOriginal: {raw_data["text"].iloc[i][:60]}')
    print(f'Cleaned:  {cleaned_data["text"].iloc[i][:60]}')

Cleaning text...
After cleaning: 19857 rows

Before vs After:

Original: کبھی کبھار ہی خیالی پلاو بناتا ہوں
Cleaned:  کبھی کبھار ہی خیالی پلاو بناتا ہوں

Original: اور پھر ممکن ہے کہ پاکستان بھی ہو
Cleaned:  اور پھر ممکن ہے کہ پاکستان بھی ہو

Original: یہ فیصلہ بھی گزشتہ دو سال میں
Cleaned:  یہ فیصلہ بھی گزشتہ دو سال میں


## Part 3: Making Pairs

For chatbot we need input-output pairs. We're using adjacent sentences:
- sentence i = input
- sentence i+1 = output

In [10]:
MAX_TOKENS = 96

all_sentences = cleaned_data['text'].tolist()


sentence_pairs = []
for i in range(len(all_sentences) - 1):
    src = all_sentences[i]
    tgt = all_sentences[i + 1]
    sentence_pairs.append((src, tgt))


filtered_pairs = []
for src, tgt in sentence_pairs:
    src_words = len(src.split())
    tgt_words = len(tgt.split())
    
    if 2 <= src_words <= 64 and 2 <= tgt_words <= 64:
        filtered_pairs.append((src, tgt))

print(f'Total pairs: {len(filtered_pairs)}')


# 80/10/10 
if filtered_pairs:
    sources, targets = zip(*filtered_pairs)
else:
    sources, targets = [], []

X_train, X_temp, y_train, y_temp = train_test_split(
    sources, targets, 
    test_size=0.2, 
    random_state=SEED, 
    shuffle=True
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5, 
    random_state=SEED, 
    shuffle=True
)

print(f'\nSplit:')
print(f'  Train: {len(X_train)}')
print(f'  Val:   {len(X_val)}')
print(f'  Test:  {len(X_test)}')


for i in range(3):
    print(f'\n{i+1}.')
    print(f'  In:  {X_train[i][:70]}')
    print(f'  Out: {y_train[i][:70]}')

Total pairs: 19421

Split:
  Train: 15536
  Val:   1942
  Test:  1943

1.
  In:  اوراس کے ساتھ اداروں میں تصادم کو بھی روکنا ہے.
  Out: جذباتی لوگوں کو نہیں سمجھ پاتا

2.
  In:  کہ لوگ روحانیت کے لیے قران وسنت کے بجایے
  Out: اس میں کشمیریوں کا بھی فایدہ ہے.

3.
  In:  صدرالدین نے بتایا کہ ایسا ہی ہوا.
  Out: جہاں جاییں وعظ اور نصیحت کی باتیں.


## Part 4: Tokenizer

Switched to SentencePiece (BPE algorithm) - works way better!



In [12]:
# write all our text to one file for training tokenizer
corpus_file = save_folder / 'corpus_for_tokenizer.txt'

print('Writing corpus...')
with open(corpus_file, 'w', encoding='utf-8') as f:
    for src, tgt in zip(X_train, y_train):
        f.write(src + '\n')
        f.write(tgt + '\n')
    for src, tgt in zip(X_val, y_val):
        f.write(src + '\n')
        f.write(tgt + '\n')
    for src, tgt in zip(X_test, y_test):
        f.write(src + '\n')
        f.write(tgt + '\n')

print(f'Corpus saved: {corpus_file}')


VOCAB_SIZES_TO_TRY = [16000, 12000, 8000, 4000, 2000, 1000, 800, 600, 400, 300, 200]
tokenizer_prefix = str(tokenizer_folder / 'urdu_tokenizer')


def cleanup_old_tokenizer(prefix):
    for extension in ['.model', '.vocab']:
        old_file = prefix + extension
        if os.path.exists(old_file):
            try:
                os.remove(old_file)
            except:
                pass  # ignore errors

print('\nTraining tokenizer...')
tokenizer_trained = False
last_error = None

for vocab_size in VOCAB_SIZES_TO_TRY:
    try:
        cleanup_old_tokenizer(tokenizer_prefix)
        
        spm.SentencePieceTrainer.Train(
            input=str(corpus_file),
            model_prefix=tokenizer_prefix,
            vocab_size=vocab_size,
            model_type='bpe',  # byte pair encoding
            character_coverage=0.9995,
            pad_id=0,  # special tokens
            bos_id=1,
            eos_id=2,
            unk_id=3,
            control_symbols='[SEP]'  # extra special token
        )
        
        tokenizer_trained = True
        print(f'SUCCESS! Tokenizer trained with vocab_size={vocab_size}')
        break
        
    except Exception as e:
        last_error = e
        print(f'vocab_size={vocab_size} failed: {str(e)[:60]}...')

if not tokenizer_trained:
    raise Exception(f'Could not train tokenizer :( Last error: {last_error}')

sp_tokenizer = spm.SentencePieceProcessor()
sp_tokenizer.load(tokenizer_prefix + '.model')


PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

print(f'\nTokenizer loaded!')
print(f'Vocab size: {sp_tokenizer.get_piece_size()}')
print(f'Special tokens: PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}, UNK={UNK_ID}')

# test it
test_text = 'السلام علیکم کیسے ہیں'
test_tokens = sp_tokenizer.encode(test_text, out_type=int)
print(f'\nTest: "{test_text}"')
print(f'Tokens: {test_tokens[:15]}...')

Writing corpus...
Corpus saved: /kaggle/working/my_chatbot_stuff/corpus_for_tokenizer.txt

Training tokenizer...
SUCCESS! Tokenizer trained with vocab_size=16000

Tokenizer loaded!
Vocab size: 16000
Special tokens: PAD=0, BOS=1, EOS=2, UNK=3

Test: "السلام علیکم کیسے ہیں"
Tokens: [122, 15966, 224, 57, 5379, 15964, 586, 43]...


## Part 5: Dataset Class

PyTorch needs a Dataset class.

The collate function creates batches and masks - masks tell model which parts to ignore

In [14]:
BATCH_SIZE = 64   

class UrduPairDataset(Dataset):

    
    def __init__(self, source_texts, target_texts, tokenizer, max_length=MAX_TOKENS):
        self.sources = list(source_texts)
        self.targets = list(target_texts)
        self.tokenizer = tokenizer
        self.max_len = max_length
    
    def __len__(self):
        return len(self.sources)
    
    def encode_text(self, text):
        """encode text to token ids with BOS and EOS"""
        token_ids = self.tokenizer.encode(text, out_type=int)
        
        # truncate if too long 
        if len(token_ids) > self.max_len - 2:
            token_ids = token_ids[:self.max_len - 2]
        
        # add BOS at start, EOS at end
        return torch.tensor([BOS_ID] + token_ids + [EOS_ID], dtype=torch.long)
    
    def __getitem__(self, idx):
        src_encoded = self.encode_text(self.sources[idx])
        tgt_encoded = self.encode_text(self.targets[idx])
        return src_encoded, tgt_encoded

def collate_function(batch):
    """
    batch pairs together with padding
    also creates masks for transformer
    """
    source_list, target_list = zip(*batch)
    
    # find max lengths in this batch
    max_src_len = max(len(s) for s in source_list)
    max_tgt_len = max(len(t) for t in target_list)
    
    batch_size = len(batch)
    

    src_padded = torch.full((batch_size, max_src_len), PAD_ID, dtype=torch.long)
    tgt_padded = torch.full((batch_size, max_tgt_len), PAD_ID, dtype=torch.long)
    

    for i, (src, tgt) in enumerate(zip(source_list, target_list)):
        src_padded[i, :len(src)] = src
        tgt_padded[i, :len(tgt)] = tgt
    
    # for decoder: input is everything except last token
    # output is everything except first token (shifted by 1)
    decoder_input = tgt_padded[:, :-1]
    decoder_output = tgt_padded[:, 1:]
    
    # masks: True means "ignore this position"
    src_padding_mask = (src_padded == PAD_ID)
    tgt_padding_mask = (decoder_input == PAD_ID)
    
    # causal mask: prevents looking at future tokens
    seq_len = decoder_input.size(1)
    causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
    
    return src_padded, decoder_input, decoder_output, src_padding_mask, tgt_padding_mask, causal_mask


print('Creating datasets...')
train_dataset = UrduPairDataset(X_train, y_train, sp_tokenizer)
val_dataset = UrduPairDataset(X_val, y_val, sp_tokenizer)
test_dataset = UrduPairDataset(X_test, y_test, sp_tokenizer)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_function
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_function
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_function
)

print(f'Done!')
print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

Creating datasets...
Done!
Train batches: 243
Val batches:   31
Test batches:  31


## Part 6: Transformer Components (Built from Scratch!)

This took FOREVER. We're building everything ourselves:
- Multi-head attention (the hard part)
- Positional encoding 
- Feed forward networks
- Encoder layers
- Decoder layers
- Full transformer

Prof said we HAVE to build from scratch, can't just use nn.Transformer :(
So here we go...

In [15]:

# this uses sine and cosine to add position info
class PositionalEncoding(nn.Module):
   
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
    
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        
 
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * 
            (-math.log(10000.0) / d_model)
        )
        
        # sine for even dimensions, cosine for odd
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  
        self.register_buffer('pe', pe)  
    
    def forward(self, x):
      
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

print('Positional encoding done')

Positional encoding done


In [19]:
# Multihead attention

class MultiHeadAttention(nn.Module):
  
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension of each head
        
        # linear layers for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # output projection
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def split_heads(self, x):
      
        batch_size, seq_len, d_model = x.size()
        # reshape to (batch, seq_len, num_heads, d_k)
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        # transpose to (batch, num_heads, seq_len, d_k)
        return x.transpose(1, 2)
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
    
        # Q, K, V shape: (batch, num_heads, seq_len, d_k)
        
        # compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # apply attention to values
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # linear projections
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        
        # apply attention
        attn_output, _ = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, -1, self.d_model)
        
        # final linear projection
        output = self.W_o(attn_output)
        
        return output

print('Multi-head attentin')

Multi-head attentin


In [20]:
# simple 2-layer MLP with ReLU
class FeedForward(nn.Module):
    """position-wise feed-forward network"""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x -> linear -> relu -> dropout -> linear
        x = F.relu(self.linear1(x))
        x = self.dropout(x)
        x = self.linear2(x)
        return x

print('Feed forward network done')

Feed forward network done


In [21]:
# Encoder Layer
# self-attention + feed-forward with residual connections
class EncoderLayer(nn.Module):
    """single transformer encoder layer"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # self attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # feed forward
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        # batch normalization instead of layer norm 
        self.bn1 = nn.BatchNorm1d(d_model)
        self.bn2 = nn.BatchNorm1d(d_model)
        
        # dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # self attention with residual connection
        attn_output = self.self_attention(x, x, x, mask)
        # batch norm needs (batch, features, seq) so transpose
        x_res = x + self.dropout1(attn_output)
        x = self.bn1(x_res.transpose(1, 2)).transpose(1, 2)
        
        # feed forward with residual connection
        ff_output = self.feed_forward(x)
        x_res = x + self.dropout2(ff_output)
        x = self.bn2(x_res.transpose(1, 2)).transpose(1, 2)
        
        return x

print('Encoder layer done')

Encoder layer done


In [22]:
# Decoder Layer  
# self-attention + cross-attention + feed-forward
class DecoderLayer(nn.Module):
    """single transformer decoder layer"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # self attention (masked)
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # cross attention (attend to encoder output)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # feed forward
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        # batch normalization
        self.bn1 = nn.BatchNorm1d(d_model)
        self.bn2 = nn.BatchNorm1d(d_model)
        self.bn3 = nn.BatchNorm1d(d_model)
        
        # dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # masked self attention
        self_attn_output = self.self_attention(x, x, x, tgt_mask)
        x_res = x + self.dropout1(self_attn_output)
        x = self.bn1(x_res.transpose(1, 2)).transpose(1, 2)
        
        # cross attention to encoder output
        cross_attn_output = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x_res = x + self.dropout2(cross_attn_output)
        x = self.bn2(x_res.transpose(1, 2)).transpose(1, 2)
        
        # feed forward
        ff_output = self.feed_forward(x)
        x_res = x + self.dropout3(ff_output)
        x = self.bn3(x_res.transpose(1, 2)).transpose(1, 2)
        
        return x

print('Decoder layer done')

Decoder layer done


In [23]:

# putting it all together!
class UrduChatbotTransformer(nn.Module):
    """complete transformer encoder-decoder"""
    
    def __init__(self, vocab_size, d_model=256, num_heads=8, d_ff=1024,
                 num_encoder_layers=4, num_decoder_layers=4, max_len=512,
                 dropout=0.1, pad_idx=0):
        super().__init__()
        
        self.d_model = d_model
        self.pad_idx = pad_idx
        
        # embeddings
        self.encoder_embedding = nn.Embedding(vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(vocab_size, d_model)
        
        # positional encodings
        self.encoder_pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.decoder_pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # encoder layers
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])
        
        # decoder layers
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])
        
        # output projection
        self.output_projection = nn.Linear(d_model, vocab_size)
        
        # initialize parameters
        self._init_parameters()
    
    def _init_parameters(self):
        
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def make_src_mask(self, src):
        
        # src shape: (batch, src_len)
        src_mask = (src != self.pad_idx).unsqueeze(1).unsqueeze(2)
        # shape: (batch, 1, 1, src_len)
        return src_mask
    
    def make_tgt_mask(self, tgt):
      
        batch_size, tgt_len = tgt.size()
        
        # padding mask
        tgt_pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        # shape: (batch, 1, 1, tgt_len)
        
        # causal mask (lower triangular)
        tgt_sub_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
        # shape: (tgt_len, tgt_len)
        
        # combine masks
        tgt_mask = tgt_pad_mask & tgt_sub_mask
        return tgt_mask
    
    def forward(self, src, tgt):
        # create masks
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        
        # encoder
        x = self.encoder_embedding(src) * math.sqrt(self.d_model)
        x = self.encoder_pos_encoding(x)
        
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        
        encoder_output = x
        
        # decoder
        x = self.decoder_embedding(tgt) * math.sqrt(self.d_model)
        x = self.decoder_pos_encoding(x)
        
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        
        # project to vocabulary
        output = self.output_projection(x)
        
        return output

print('Full transformer')
print('All components built from scratch')

Full transformer
All components built from scratch


In [24]:

print('Creating model...')
model_config = {
    'vocab_size': sp_tokenizer.get_piece_size(),
    'd_model': 256,  # to train faster
    'num_heads': 8,  
    'd_ff': 1024,  #ff
    'num_encoder_layers': 4,
    'num_decoder_layers': 4,
    'max_len': 512,
    'dropout': 0.1,
    'pad_idx': PAD_ID
}

model = UrduChatbotTransformer(**model_config).to(device)


total_params = sum(p.numel() for p in model.parameters())
print(f'Model created!')
print(f'Total parameters: {total_params:,} ({total_params/1e6:.2f}M)')
print(f'Config: {model_config}')

Creating model...
Model created!
Total parameters: 19,676,800 (19.68M)
Config: {'vocab_size': 16000, 'd_model': 256, 'num_heads': 8, 'd_ff': 1024, 'num_encoder_layers': 4, 'num_decoder_layers': 4, 'max_len': 512, 'dropout': 0.1, 'pad_idx': 0}


## Part 7: Training Setup

Loss function with label smoothing (makes model less overconfident)
Adam optimizer - standard choice for transformers
Gradient clipping to prevent exploding gradients

In [25]:
# label smoothing loss
# prevents model from being too confident about predictions
class LabelSmoothingLoss(nn.Module):
    def __init__(self, num_classes, smoothing=0.1, ignore_index=PAD_ID):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.ignore_index = ignore_index
    
    def forward(self, pred, target):
        # pred shape: (batch, seq_len, vocab_size)
        # target shape: (batch, seq_len)
        
        batch_size, seq_len, vocab_size = pred.shape
        
        pred = pred.reshape(-1, vocab_size)
        target = target.reshape(-1)
        
        mask = (target != self.ignore_index)
        pred = pred[mask]
        target = target[mask]
        
        with torch.no_grad():
            true_dist = torch.full_like(pred, self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        
        log_probs = F.log_softmax(pred, dim=-1)
        loss = (-true_dist * log_probs).sum(dim=-1).mean()
        
        return loss

criterion = LabelSmoothingLoss(
    num_classes=sp_tokenizer.get_piece_size(),
    smoothing=0.1,
    ignore_index=PAD_ID
)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
GRAD_CLIP = 1.0  
EPOCHS = 20

print('Training setup complete')
print(f'Optimizer: Adam (lr=3e-4)')
print(f'Loss: Label Smoothing CrossEntropy')
print(f'Epochs: {EPOCHS}')

Training setup complete
Optimizer: Adam (lr=3e-4)
Loss: Label Smoothing CrossEntropy
Epochs: 20


In [26]:
# decoding functions - for generating text

@torch.no_grad()
def generate_greedy(input_text, max_tokens=64):
    """generate response using greedy decoding"""
    model.eval()
    
    # clean and encode input
    tokens = sp_tokenizer.encode(clean_urdu_text(input_text), out_type=int)
    if len(tokens) > MAX_TOKENS - 2:
        tokens = tokens[:MAX_TOKENS - 2]
    
    # add BOS and EOS
    src = torch.tensor([[BOS_ID] + tokens + [EOS_ID]], device=device)
    
    # start decoding with BOS
    generated = torch.tensor([[BOS_ID]], device=device)
    
    # generate one token at a time
    for _ in range(max_tokens):
        # forward pass
        logits = model(src, generated)
        
        # get next token (highest probability)
        next_token = logits[:, -1, :].argmax(-1)
        
        # add to sequence
        generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
        
        # stop if EOS
        if next_token.item() == EOS_ID:
            break
    
    # decode to text
    output_ids = generated[0, 1:].tolist()  # remove BOS
    
    # remove EOS if present
    if EOS_ID in output_ids:
        eos_pos = output_ids.index(EOS_ID)
        output_ids = output_ids[:eos_pos]
    
    return sp_tokenizer.decode(output_ids)

# quick BLEU score calculation for validation
@torch.no_grad()
def calculate_bleu(sources, targets, num_samples=200):
    model.eval()
    
    num_samples = min(num_samples, len(sources))
    references = []
    hypotheses = []
    
    for i in range(num_samples):
        hyp = generate_greedy(sources[i])
        hypotheses.append(hyp)
        references.append([targets[i]])
    
    if not hypotheses:
        return 0.0
    
    bleu = sacrebleu.corpus_bleu(hypotheses, list(zip(*references))[0])
    return bleu.score

print('Decoding functions ready')

Decoding functions ready


In [27]:
def train_one_epoch(dataloader, is_training=True):
    """run one epoch"""
    if is_training:
        model.train()
    else:
        model.eval()
    
    total_loss = 0.0
    total_tokens = 0
    correct = 0
    total_preds = 0
    
    for batch in dataloader:
        src, tgt_in, tgt_out, _, _, _ = batch
        
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)
        
        if is_training:
            optimizer.zero_grad()
        
        # forward
        logits = model(src, tgt_in)
        loss = criterion(logits, tgt_out)
        
        if is_training:
            # backward
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        
        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            mask = (tgt_out != PAD_ID)
            correct += ((preds == tgt_out) & mask).sum().item()
            total_preds += mask.sum().item()
        
        n_tokens = (tgt_out != PAD_ID).sum().item()
        total_loss += loss.item() * max(1, n_tokens)
        total_tokens += max(1, n_tokens)
    
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(avg_loss) if avg_loss < 50 else float('inf')
    acc = correct / max(1, total_preds)
    
    return avg_loss, ppl, acc

print('Training functions ready')

Training functions ready


In [28]:
checkpoint_path = save_folder / 'best_model.pt'
best_val_bleu = -1.0



for epoch in range(1, EPOCHS + 1):
    
    train_loss, train_ppl, train_acc = train_one_epoch(train_loader, is_training=True)
    
    val_loss, val_ppl, val_acc = train_one_epoch(val_loader, is_training=False)
    
    if epoch % 2 == 0 or epoch == 1:
        val_bleu = calculate_bleu(list(X_val), list(y_val), num_samples=200)
    else:
        val_bleu = best_val_bleu  
    
    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'train_ppl={train_ppl:6.2f} train_acc={train_acc*100:5.2f}% | '
          f'val_ppl={val_ppl:6.2f} val_acc={val_acc*100:5.2f}% | '
          f'val_BLEU={val_bleu:5.2f}')
    
    if val_bleu > best_val_bleu:
        best_val_bleu = val_bleu
        
        # save checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config': model_config,
            'vocab_size': sp_tokenizer.get_piece_size(),
            'best_bleu': best_val_bleu
        }, checkpoint_path)
        
        print(f'  ** New best! Saved to {checkpoint_path}')


print(f'Best validation BLEU: {best_val_bleu:.2f}')


Epoch 01/20 | train_ppl=1294.34 train_acc=11.66% | val_ppl=629.46 val_acc=15.63% | val_BLEU= 0.00
  ** New best! Saved to /kaggle/working/my_chatbot_stuff/best_model.pt
Epoch 02/20 | train_ppl=540.88 train_acc=16.11% | val_ppl=476.27 val_acc=17.65% | val_BLEU= 0.59
  ** New best! Saved to /kaggle/working/my_chatbot_stuff/best_model.pt
Epoch 03/20 | train_ppl=392.83 train_acc=19.12% | val_ppl=387.08 val_acc=19.96% | val_BLEU= 0.59
Epoch 04/20 | train_ppl=297.11 train_acc=22.19% | val_ppl=325.59 val_acc=22.37% | val_BLEU= 0.83
  ** New best! Saved to /kaggle/working/my_chatbot_stuff/best_model.pt
Epoch 05/20 | train_ppl=228.02 train_acc=25.09% | val_ppl=284.69 val_acc=24.42% | val_BLEU= 0.83
Epoch 06/20 | train_ppl=173.89 train_acc=28.37% | val_ppl=250.26 val_acc=26.52% | val_BLEU= 0.59
Epoch 07/20 | train_ppl=133.64 train_acc=31.79% | val_ppl=215.45 val_acc=29.11% | val_BLEU= 0.83
Epoch 08/20 | train_ppl=101.93 train_acc=35.89% | val_ppl=197.56 val_acc=31.14% | val_BLEU= 0.00
Epoch 09/2

In [31]:
# load the best model we saved during training
print('Loading best model...')
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print('Model loaded!')

# function to test on a bunch of examples
@torch.no_grad()
def test_on_dataset(sources, targets, num_samples=200):
    """test model and calculate metrics"""
    model.eval()
    
    num_samples = min(num_samples, len(sources))
    hypotheses = []
    references = []
    
    print(f'Generating {num_samples} responses...')
    for i in range(num_samples):
        # generate response for this input
        hyp = generate_greedy(sources[i])
        hypotheses.append(hyp)
        references.append([targets[i]])
        
        if (i + 1) % 50 == 0:
            print(f'  done {i+1}/{num_samples}')
    
    # calculate BLEU score
    bleu = sacrebleu.corpus_bleu(hypotheses, list(zip(*references))[0])
    
    # chrF score (character-level, better for urdu)
    chrf = sacrebleu.corpus_chrf(hypotheses, list(zip(*references))[0])
    
    # ROUGE-L (longest common subsequence)
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    rouge_scores = []
    for hyp, ref in zip(hypotheses, references):
        score = rouge.score(ref[0], hyp)['rougeL'].fmeasure
        rouge_scores.append(score)
    rouge_l = float(np.mean(rouge_scores))
    
    # get perplexity from actual forward pass
    temp_loader = DataLoader(
        UrduPairDataset(sources[:num_samples], targets[:num_samples], sp_tokenizer),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_function
    )
    _, ppl, acc = train_one_epoch(temp_loader, is_training=False)
    
    return {
        'BLEU': round(bleu.score, 2),
        'chrF': round(chrf.score, 2),
        'ROUGE-L': round(rouge_l, 3),
        'Perplexity': round(ppl, 2),
        'Accuracy': round(acc * 100, 2)
    }

# run evaluation
print('\nTesting on test set...')
test_metrics = test_on_dataset(list(X_test), list(y_test), num_samples=min(300, len(X_test)))

print('\n' + '='*60)
print('FINAL TEST RESULTS:')
print('='*60)
for metric, value in test_metrics.items():
    print(f'{metric:12s}: {value}')
print('='*60)

Loading best model...
Model loaded!

Testing on test set...
Generating 300 responses...
  done 50/300
  done 100/300
  done 150/300
  done 200/300
  done 250/300
  done 300/300

FINAL TEST RESULTS:
BLEU        : 0.0
chrF        : 32.0
ROUGE-L     : 0.0
Perplexity  : 327.77
Accuracy    : 22.73


In [32]:
# show some example outputs
print('\nExample Outputs:')
print('='*80)

for i in range(min(5, len(X_test))):
    print(f'\n{i+1}.')
    print(f'  Input:     {X_test[i]}')
    print(f'  Reference: {y_test[i]}')
    print(f'  Generated: {generate_greedy(X_test[i])}')

print('\n' + '='*80)


Example Outputs:

1.
  Input:     ایم کیو ایم کرنا پریس کانفرنس دیکھیے سوشل میڈیا پر دستیاب ہونا
  Reference: دوا ءبھی شفا ءاسی کے حکم سے بنتی ہے
  Generated: اس سے یہ نہیں.

2.
  Input:     بنگلہ دیش کا موسم گرما کا وقت
  Reference: مصباح الحق کا نام تیز ترین ٹیسٹ سنچری بنانے والوںمیں شامل
  Generated: یہ ایک بات ہے.

3.
  Input:     قطعی طور پر نشوزنہیں سمجھی جایے گی
  Reference: سو تیر ترازو تھے دل میں جب ہم نے رقص اغاز کیا
  Generated: اس سے یہ نہیں.

4.
  Input:     اور چھ ماہ تک چترال کا رابطہ ملک کے دوسرے حصوں سے منقطع رہتا ہے .
  Reference: الہن نیاز ایک نوجوان تاریخ دان ہیں.
  Generated: اس سے یہ بھی ایک بات ہے.

5.
  Input:     میری پینٹ بار بار نیچے ہو رہی ہے
  Reference: جن کا ماضی نہ ہو, کیا ان کا کویی مستقبل ہوتا ہے?
  Generated: اس سے یہ نہیں.



In [35]:
pip install gradio


Note: you may need to restart the kernel to use updated packages.


In [40]:
import torch

def load_model_and_tokenizer(checkpoint_path):
    """Load trained Urdu chatbot model and tokenizer"""
    print("Loading model and tokenizer...")
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print("Model loaded successfully!")
    return model, sp_tokenizer


@torch.no_grad()
def chat_with_model(user_input):
    """Generate chatbot response for a given Urdu input"""
    if not user_input.strip():
        return "براہ کرم کوئی سوال درج کریں۔"  # "Please enter a question."
    
    # Generate model response
    response = generate_greedy(user_input)
    return response


In [41]:
# Load model once
model, tokenizer = load_model_and_tokenizer(checkpoint_path)

# Chat with the model interactively
user_text = "آپ کا نام کیا ہے؟"
print("User:", user_text)
print("Bot:", chat_with_model(user_text))


Loading model and tokenizer...
Model loaded successfully!
User: آپ کا نام کیا ہے؟
Bot: اس سے یہ نہیں.


In [42]:
import gradio as gr

# Gradio interface
demo = gr.Interface(
    fn=chat_with_model,
    inputs=gr.Textbox(lines=2, placeholder="اپنا سوال یہاں لکھیں..."),
    outputs=gr.Textbox(label="جواب"),
    title="اردو چیٹ بوٹ",
    description="ایک اردو چیٹ بوٹ جو آپ کے سوالات کے جوابات دیتا ہے۔"
)

demo.launch()


Running on local URL:  http://127.0.0.1:7861
Kaggle notebooks require sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Running on public URL: https://d32f6ae7a78b341022.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
